In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_path = "../Online Retail.xlsx"

df = pd.read_excel(file_path)

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique()

In [ ]:
(df["Quantity"] < 0).sum()

In [ ]:
(df["UnitPrice"] <= 0).sum()

In [ ]:
(df["UnitPrice"] <= 0).sum()

In [ ]:
df["Description"].isnull().sum()

In [ ]:
df["InvoiceDate"].min(), df["InvoiceDate"].max()

In [ ]:
df["CustomerID"].nunique()

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean = df_clean.drop_duplicates()

In [ ]:
df_clean.shape

In [ ]:
df_clean.duplicated().sum()

In [ ]:
df_clean["CustomerID"].isnull().sum()

In [ ]:
customer_df = df_clean.dropna(subset=["CustomerID"]).copy()

In [ ]:
customer_df.shape

In [ ]:
(customer_df["Quantity"] < 0).sum()

In [ ]:
sales_df = customer_df[customer_df["Quantity"] > 0].copy()

In [ ]:
sales_df.shape

In [ ]:
(sales_df["UnitPrice"] <= 0).sum()

In [ ]:
sales_df = sales_df[sales_df["UnitPrice"] > 0].copy()

In [ ]:
sales_df.shape

In [ ]:
sales_df["Revenue"] = sales_df["Quantity"] * sales_df["UnitPrice"]

In [ ]:
sales_df[["Quantity", "UnitPrice", "Revenue"]].head()

In [ ]:
sales_df["Revenue"].describe()

In [ ]:
sales_df["Revenue"].sum()

In [ ]:
import pandas as pd

file_path = "../Online Retail.xlsx"
df = pd.read_excel(file_path)

df_clean = df.drop_duplicates()

customer_df = df_clean.dropna(subset=["CustomerID"]).copy()

sales_df = customer_df[
    (customer_df["Quantity"] > 0) &
    (customer_df["UnitPrice"] > 0)
].copy()

sales_df["Revenue"] = sales_df["Quantity"] * sales_df["UnitPrice"]

In [ ]:
sales_df["Month"] = sales_df["InvoiceDate"].dt.to_period("M")

In [ ]:
monthly_revenue = sales_df.groupby("Month")["Revenue"].sum()

In [ ]:
monthly_revenue

In [ ]:
country_revenue = sales_df.groupby("Country")["Revenue"].sum().sort_values(ascending=False)

In [ ]:
country_revenue.head(10)

In [ ]:
product_revenue = (
    sales_df.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
product_revenue.head(10)

In [ ]:
customer_revenue = (
    sales_df.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
customer_revenue.head(10)

In [ ]:
sales_df["InvoiceDate"].max()

In [ ]:
reference_date = sales_df["InvoiceDate"].max()

recency = sales_df.groupby("CustomerID")["InvoiceDate"].max().apply(
    lambda x: (reference_date - x).days
)

In [ ]:
recency.head()

In [ ]:
frequency = sales_df.groupby("CustomerID")["InvoiceNo"].nunique()

In [ ]:
frequency.head()

In [ ]:
monetary = sales_df.groupby("CustomerID")["Revenue"].sum()

In [ ]:
monetary.head()

In [ ]:
rfm = pd.concat([recency, frequency, monetary], axis=1)

rfm.columns = ["Recency", "Frequency", "Monetary"]

In [ ]:
rfm.head()

In [ ]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)

In [ ]:
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

In [ ]:
rfm[["Frequency", "F_Score"]].head(10)

In [ ]:
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

In [ ]:
rfm[["Monetary", "M_Score"]].head(10)

In [ ]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(int).astype(str) +
    rfm["F_Score"].astype(int).astype(str) +
    rfm["M_Score"].astype(int).astype(str)
)

In [ ]:
rfm.head(10)

In [ ]:
rfm["Segment"] = "Other"

rfm.loc[
    (rfm["R_Score"] >= 4) &
    (rfm["F_Score"] >= 4) &
    (rfm["M_Score"] >= 4),
    "Segment"
] = "Champions"

In [ ]:
rfm["Segment"].value_counts()

In [ ]:
rfm.loc[
    (rfm["R_Score"] >= 3) &
    (rfm["F_Score"] >= 4) &
    (rfm["M_Score"] >= 3) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Loyal Customers"

In [ ]:
rfm["Segment"].value_counts()

In [ ]:
rfm.loc[
    (rfm["R_Score"] >= 4) &
    (rfm["F_Score"] >= 2) &
    (rfm["M_Score"] >= 2) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Potential Loyalists"

In [ ]:
rfm["Segment"].value_counts()

In [ ]:
rfm.loc[
    (rfm["R_Score"] <= 2) &
    (rfm["F_Score"] >= 2) &
    (rfm["M_Score"] >= 2) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "At Risk"

In [ ]:
rfm["Segment"].value_counts()

In [ ]:
rfm.loc[
    (rfm["R_Score"] <= 2) &
    (rfm["F_Score"] <= 2) &
    (rfm["M_Score"] <= 2) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Lost Customers"

In [ ]:
rfm["Segment"].value_counts()

In [ ]:
segment_revenue = rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)

In [ ]:
segment_revenue

In [ ]:
segment_summary = rfm.groupby("Segment").agg(
    Customers=("Monetary", "count"),
    Total_Revenue=("Monetary", "sum"),
    Avg_Revenue=("Monetary", "mean"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean")
).sort_values("Total_Revenue", ascending=False)

In [ ]:
segment_summary

In [ ]:
rfm.to_csv("rfm_customer_segments.csv")

In [ ]:
rfm["Segment"] = "Other"

In [ ]:
rfm.loc[
    (rfm["R_Score"] >= 4) &
    (rfm["F_Score"] >= 4) &
    (rfm["M_Score"] >= 4),
    "Segment"
] = "Champions"

In [ ]:
rfm.loc[
    (rfm["R_Score"] >= 3) &
    (rfm["F_Score"] >= 4) &
    (rfm["M_Score"] >= 3) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Loyal Customers"

In [ ]:
rfm.loc[
    (rfm["R_Score"] >= 4) &
    (rfm["F_Score"] >= 2) &
    (rfm["M_Score"] >= 2) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Potential Loyalists"

In [ ]:
rfm.loc[
    (rfm["R_Score"] <= 2) &
    (rfm["F_Score"] >= 3) &
    (rfm["M_Score"] >= 3) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "At Risk"

In [ ]:
rfm.loc[
    (rfm["R_Score"] <= 2) &
    (rfm["F_Score"] <= 2) &
    (rfm["M_Score"] <= 2) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Lost Customers"

In [ ]:
rfm["Segment"].value_counts()

In [ ]:
final_segment_revenue = (
    rfm.groupby("Segment")["Monetary"]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
final_segment_revenue

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel("../Online Retail.xlsx")

df_clean = df.drop_duplicates()

customer_df = df_clean.dropna(subset=["CustomerID"]).copy()

sales_df = customer_df[customer_df["Quantity"] > 0].copy()

sales_df = sales_df[sales_df["UnitPrice"] > 0].copy()

sales_df["Revenue"] = sales_df["Quantity"] * sales_df["UnitPrice"]

sales_df.shape

In [ ]:
reference_date = sales_df["InvoiceDate"].max()

recency = sales_df.groupby("CustomerID")["InvoiceDate"].max().apply(
    lambda x: (reference_date - x).days
)

recency.head()

In [ ]:
frequency = sales_df.groupby("CustomerID")["InvoiceNo"].nunique()

frequency.head()

In [ ]:
monetary = sales_df.groupby("CustomerID")["Revenue"].sum()

monetary.head()

In [ ]:
rfm = pd.concat([recency, frequency, monetary], axis=1)

rfm.columns = ["Recency", "Frequency", "Monetary"]

rfm.head()

In [ ]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm[["Recency", "R_Score"]].head()

In [ ]:
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm[["Frequency", "F_Score"]].head()

In [ ]:
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm[["Monetary", "M_Score"]].head()

In [ ]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(int).astype(str) +
    rfm["F_Score"].astype(int).astype(str) +
    rfm["M_Score"].astype(int).astype(str)
)

rfm[["Recency", "Frequency", "Monetary", "RFM_Score"]].head()

In [ ]:
rfm["Segment"] = "Other"

rfm.loc[
    (rfm["R_Score"] >= 4) &
    (rfm["F_Score"] >= 4) &
    (rfm["M_Score"] >= 4),
    "Segment"
] = "Champions"

rfm.loc[
    (rfm["R_Score"] >= 3) &
    (rfm["F_Score"] >= 4) &
    (rfm["M_Score"] >= 3) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Loyal Customers"

rfm.loc[
    (rfm["R_Score"] >= 4) &
    (rfm["F_Score"] >= 2) &
    (rfm["M_Score"] >= 2) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Potential Loyalists"

rfm.loc[
    (rfm["R_Score"] <= 2) &
    (rfm["F_Score"] >= 3) &
    (rfm["M_Score"] >= 3) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "At Risk"

rfm.loc[
    (rfm["R_Score"] <= 2) &
    (rfm["F_Score"] <= 2) &
    (rfm["M_Score"] <= 2) &
    (rfm["Segment"] == "Other"),
    "Segment"
] = "Lost Customers"

rfm[["Recency", "Frequency", "Monetary", "RFM_Score", "Segment"]].head(10)

In [ ]:
segment_counts = rfm["Segment"].value_counts()

segment_counts

In [ ]:
segment_revenue = (
    rfm.groupby("Segment")["Monetary"]
    .sum()
    .sort_values(ascending=False)
)

segment_revenue

In [ ]:
segment_percentage = (
    segment_revenue / segment_revenue.sum() * 100
).round(2)

segment_percentage

In [ ]:
aov = sales_df["Revenue"].sum() / sales_df["InvoiceNo"].nunique()

aov

In [ ]:
rfm.to_csv("rfm_customer_segments.csv", index=True)

In [ ]:
customer_orders = sales_df.groupby("CustomerID")["InvoiceNo"].nunique()

repeat_customers = (customer_orders > 1).sum()
one_time_customers = (customer_orders == 1).sum()

repeat_customers, one_time_customers

In [ ]:
repeat_percentage = round(repeat_customers / (repeat_customers + one_time_customers) * 100, 2)
one_time_percentage = round(one_time_customers / (repeat_customers + one_time_customers) * 100, 2)

repeat_percentage, one_time_percentage

In [ ]:
sales_df.to_csv("online_retail_sales.csv", index=False)